# 자료 준비와 기기 범위

In [1]:
from pathlib import Path
import json, sys
import pandas as pd
try:
    from IPython.display import display, HTML
except ImportError:
    class HTML:
        def __init__(self, data): self.data = data
        def _repr_html_(self): return self.data
    def display(value): print(value)

working_dir = Path.cwd().resolve()
candidates = [working_dir, working_dir / 'code', working_dir.parent, working_dir.parent / 'code']
CODE_ROOT = next((candidate for candidate in candidates if (candidate / 'rag').is_dir() and (candidate / 'notebooks').is_dir()), None)
if CODE_ROOT is None:
    raise FileNotFoundError('프로젝트 폴더, code/ 또는 code/notebooks/에서 실행하세요.')
sys.path.insert(0, str(CODE_ROOT))

RUN_LOCAL = False  # True로 바꾸기 전 입력 자료·모델·장치·저장 위치를 확인
print('기본 모드: 셀에 명시한 저장된 평가 집계 확인. API 호출·모델 다운로드·학습 없음.')

기본 모드: 셀에 명시한 저장된 평가 집계 확인. API 호출·모델 다운로드·학습 없음.


## 자료 구성

In [2]:
# 저장된 평가 집계와 실행 설정. 원본 값을 그대로 옮겼으며 새로 평가하지 않음.
data_summary = {'user_manuals': 56,
 'repair_guides': 64,
 'documents': 120,
 'chunks': 13218,
 'model_codes': 201,
 'shared_manual_groups': 103,
 'groups_by_category': {'phone': 55, 'tablet': 21, 'book': 16, 'watch': 8, 'buds': 3}}
display(pd.Series({k:v for k,v in data_summary.items() if k != 'groups_by_category'}, name='count').to_frame())
display(pd.Series(data_summary['groups_by_category'], name='shared_manual_groups').to_frame())

,count
user_manuals,56
repair_guides,64
documents,120
chunks,13218
model_codes,201
shared_manual_groups,103


,shared_manual_groups
phone,55
tablet,21
book,16
watch,8
buds,3


## 기기 필터 진단

In [3]:
# 저장된 평가 집계와 실행 설정. 원본 값을 그대로 옮겼으며 새로 평가하지 않음.
filter_scores = {'off|rerank_ft_oof': {'n_questions': 154,
                       'top1_off_rate': 0.8311688311688312,
                       'top5': {'n': 770,
                                'off': 647,
                                'off_rate': 0.8402597402597403,
                                'off_same_text': 150,
                                'same_text_share_of_off': 0.23183925811437403}},
 'on|rerank_ft_oof': {'n_questions': 154,
                      'top1_off_rate': 0.0,
                      'top5': {'n': 770,
                               'off': 0,
                               'off_rate': 0.0,
                               'off_same_text': 0,
                               'same_text_share_of_off': None}}}
display(pd.DataFrame([{'condition': k, 'questions': v['n_questions'], **v['top5']} for k,v in filter_scores.items()]).set_index('condition'))

,questions,n,off,off_rate,off_same_text,same_text_share_of_off
condition,,,,,,
off|rerank_ft_oof,154,770,647,0.84026,150,0.231839
on|rerank_ft_oof,154,770,0,0.00000,0,NaN


## 선택 실행: 로컬 PDF → 절 → 청크

In [4]:
PDF_PATH = None  # Path('직접 확보한 PDF 경로')
if RUN_LOCAL:
    from rag.pdf_reader import PdfManualLoader
    from rag.split_text import chunk_manual
    from rag.settings import ManualChunkConfig
    if PDF_PATH is None or not Path(PDF_PATH).is_file():
        raise FileNotFoundError('직접 확보한 PDF_PATH를 지정하세요.')
    loaded_documents = list(PdfManualLoader().load(PDF_PATH, doc_meta={'doc_type': 'user_manual'}))
    local_chunks = [chunk for document in loaded_documents for chunk in chunk_manual(document, ManualChunkConfig())]
    display(pd.DataFrame([{'documents': len(loaded_documents), 'chunks': len(local_chunks)}]))
else:
    print('로컬 PDF 처리를 실행하지 않았습니다.')

로컬 PDF 처리를 실행하지 않았습니다.
